In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
df = pd.read_csv('/content/drive/MyDrive/monkeytype/mt_dataset_pref4feat16.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4675 entries, 0 to 4674
Columns: 164 entries, x1_linspace to x4_f15
dtypes: float64(160), int64(4)
memory usage: 5.8 MB


In [3]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# -----------------------------
# 1) Колонки
# -----------------------------
feat_names = [
    #"linspace",
    "linspace_session_id",
    "linspace_within_session",
    #"brightness",
    "time_reaction","omission_percent",
    #"symbol_m", "symbol_c", "symbol_s", "symbol_y", "symbol_f", "symbol_j",
    "location_0", "location_1", "location_2", "location_3",
    "participant_J", "participant_F", "participant_Y",
    #"correct_symbol_m", "correct_symbol_c", "correct_symbol_s", "correct_symbol_y", "correct_symbol_f", "correct_symbol_j",
]
latent_dim = 16
feat_names_fs = [f"f{i}" for i in range(latent_dim)]
feat_names.extend(feat_names_fs)

F_DIM = len(feat_names)

pj = feat_names.index("participant_J")
pf = feat_names.index("participant_F")
py = feat_names.index("participant_Y")
participant_idx = (pj, pf, py)

def cols(prefix: str):
    return [f"{prefix}_{f}" for f in feat_names]

x1_cols = cols("x1")
x2_cols = cols("x2")
x3_cols = cols("x3")
x4_cols = cols("x4")


def prepare_arrays(df: pd.DataFrame):
    needed = x1_cols + x2_cols + x3_cols + x4_cols
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    X1 = df[x1_cols].to_numpy(dtype=np.float32)
    X2 = df[x2_cols].to_numpy(dtype=np.float32)
    X3 = df[x3_cols].to_numpy(dtype=np.float32)
    X4 = df[x4_cols].to_numpy(dtype=np.float32)
    return X1, X2, X3, X4


# -----------------------------
# 2) Dataset: возвращает четверку
# -----------------------------
class QuadDataset(Dataset):
    def __init__(self, X1, X2, X3, X4):
        self.X1 = torch.from_numpy(X1).float()
        self.X2 = torch.from_numpy(X2).float()
        self.X3 = torch.from_numpy(X3).float()
        self.X4 = torch.from_numpy(X4).float()

    def __len__(self):
        return self.X1.shape[0]

    def __getitem__(self, idx):
        return self.X1[idx], self.X2[idx], self.X3[idx], self.X4[idx]

In [4]:
def build_reward_mlp(in_dim: int, hidden=(128, 64), dropout=0.2):
    layers = []
    d = in_dim
    for h in hidden:
        layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
        d = h
    layers += [nn.Linear(d, 1)]
    return nn.Sequential(*layers)

class HardMoEReward3(nn.Module):
    """
    3 независимых reward-сети (J/F/Y), hard routing по participant one-hot.
    Выход: s (B,) — скалярный reward.
    """
    def __init__(self, in_dim: int, participant_idx: tuple[int, int, int], hidden=(128, 64), dropout=0.2):
        super().__init__()
        self.participant_idx = participant_idx

        self.experts = nn.ModuleList([
            build_reward_mlp(in_dim, hidden=hidden, dropout=dropout),  # J
            build_reward_mlp(in_dim, hidden=hidden, dropout=dropout),  # F
            build_reward_mlp(in_dim, hidden=hidden, dropout=dropout),  # Y
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        pj, pf, py = self.participant_idx
        p = x[:, [pj, pf, py]]                 # (B,3)
        idx = p.argmax(dim=1)                  # (B,) values 0/1/2

        B = x.size(0)
        s = torch.empty(B, device=x.device, dtype=x.dtype)  # (B,)

        # hard routing: считаем только подбатчи для каждого эксперта
        for k in (0, 1, 2):
            mask = (idx == k)
            if mask.any():
                s_k = self.experts[k](x[mask]).squeeze(-1)  # (Bk,)
                s[mask] = s_k

        return s

In [5]:
def loss_listwise_top1(s1, s2, s3, s4, temperature=1.0, l2_scores=9e-4):
    """
    Listwise: -log softmax winner (x1).
    L = -s1 + logsumexp([s1,s2,s3,s4])
    """
    scores0 = torch.stack([s1, s2, s3, s4], dim=1)
    scores = torch.stack([s1, s2, s3, s4], dim=1) / temperature  # (B,4)
    # target winner index = 0
    loss = F.cross_entropy(scores, torch.zeros(scores.size(0), dtype=torch.long, device=scores.device))
    # small penalty to keep scores bounded
    loss = loss + l2_scores * (scores0**2).mean()
    return loss

In [6]:
def eval_metrics(model, loader, device, temperature=1.0):
    """
    Оценка на loader для задач top-1 из 4:
      - loss (listwise или pairwise)
      - top1_acc_soft: top-1 с корректной обработкой ties (частичный кредит 1/k)
      - top1_acc_hard: обычный argmax top-1 (может завышать при ties)
      - tie_rate: доля примеров, где максимум не уникален
      - mean_margin: mean(s1 - max(s2,s3,s4))
      - pwin_*: статистики по P(x1 wins) = softmax(scores/T)[0]
    """
    model.eval()

    total_loss = 0.0
    total = 0

    # top-1 метрики
    top1_soft_sum = 0.0
    top1_hard_correct = 0

    # tie diagnostics
    tie_count = 0

    # margin
    margin_sum = 0.0

    # P(win) stats
    p_list = []

    for x1, x2, x3, x4 in loader:
        x1, x2, x3, x4 = x1.to(device), x2.to(device), x3.to(device), x4.to(device)

        s1 = model(x1); s2 = model(x2); s3 = model(x3); s4 = model(x4)
        scores = torch.stack([s1, s2, s3, s4], dim=1)  # (B,4)

        # loss
        logits = scores / temperature
        loss = F.cross_entropy(logits, torch.zeros(logits.size(0), dtype=torch.long, device=logits.device))

        bs = scores.size(0)
        total_loss += float(loss.item()) * bs
        total += bs

        # ----- hard top1 (может врать при ties)
        pred = scores.argmax(dim=1)
        top1_hard_correct += int((pred == 0).sum().item())

        # ----- soft top1 (корректно учитывает ties)
        maxv = scores.max(dim=1, keepdim=True).values          # (B,1)
        is_max = (scores == maxv).float()                      # (B,4)
        k = is_max.sum(dim=1)                                  # (B,) сколько максимумов
        # частичный кредит: если x1 среди максимумов, то 1/k
        top1_soft_sum += float((is_max[:, 0] / k).sum().item())

        # tie rate: где максимум не уникален (k>1)
        tie_count += int((k > 1).sum().item())

        # margin: s1 - max(s2,s3,s4)
        best_loser = torch.max(scores[:, 1:], dim=1).values
        margin = (scores[:, 0] - best_loser)
        margin_sum += float(margin.mean().item()) * bs

        # P(win) согласованно с температурой
        pwin = torch.softmax(scores / temperature, dim=1)[:, 0]
        p_list.append(pwin.detach().cpu().numpy())

    p = np.concatenate(p_list) if len(p_list) else np.array([np.nan], dtype=np.float32)

    def p_stats(arr):
        return {
            "mean": float(np.mean(arr)),
            "std":  float(np.std(arr)),
            "q05":  float(np.quantile(arr, 0.05)),
            "q50":  float(np.quantile(arr, 0.50)),
            "q95":  float(np.quantile(arr, 0.95)),
            "min":  float(np.min(arr)),
            "max":  float(np.max(arr)),
        }

    return {
        "loss": total_loss / max(total, 1),
        "top1_acc_soft": top1_soft_sum / max(total, 1),
        "top1_acc_hard": top1_hard_correct / max(total, 1),
        "tie_rate": tie_count / max(total, 1),
        "mean_margin": margin_sum / max(total, 1),
        "pwin": p_stats(p),
    }
def _filter_test_by_participant_onehot(
    df_test: pd.DataFrame,
    participant: str,          # "J" / "F" / "Y"
    *,
    require_consistent=True,   # проверять, что participant одинаков для x1..x4
) -> pd.DataFrame:
    if participant not in ("J", "F", "Y"):
        raise ValueError("participant must be one of: 'J', 'F', 'Y'")

    c1 = f"x1_participant_{participant}"
    c2 = f"x2_participant_{participant}"
    c3 = f"x3_participant_{participant}"
    c4 = f"x4_participant_{participant}"

    missing = [c for c in (c1, c2, c3, c4) if c not in df_test.columns]
    if missing:
        raise ValueError(f"Missing participant one-hot columns: {missing}")

    sub = df_test[df_test[c1] == 1.0].copy()

    if require_consistent and len(sub) > 0:
        # проверяем, что по этой же букве в x2..x4 тоже 1.0
        ok = (sub[c2] == 1.0) & (sub[c3] == 1.0) & (sub[c4] == 1.0)
        # если есть несогласованные строки — выкинем их, чтобы метрика была честной
        sub = sub[ok].copy()

    return sub


@torch.no_grad()
def eval_test_for_participant(
    df_test: pd.DataFrame,
    participant: str,  # "J"/"F"/"Y"
    model,
    scaler,
    device,
    *,
    temperature=1.0,
    batch_size=256,
    require_consistent=True,
):
    """
    Считает метрики на тестовой выборке только для строк конкретного participant.

    Возвращает dict:
      - n: сколько строк в подвыборке
      - metrics: результат eval_metrics (или None если n=0)
    """
    sub = _filter_test_by_participant_onehot(
        df_test, participant, require_consistent=require_consistent
    )
    n = len(sub)
    if n == 0:
        return {"participant": participant, "n": 0, "metrics": None}

    # arrays из df
    X1, X2, X3, X4 = prepare_arrays(sub)

    # scaler (fit на train!) применяем одинаково
    def tr(x):
        return scaler.transform(x).astype(np.float32)

    X1, X2, X3, X4 = tr(X1), tr(X2), tr(X3), tr(X4)

    loader = DataLoader(QuadDataset(X1, X2, X3, X4), batch_size=batch_size, shuffle=False)
    metrics = eval_metrics(model, loader, device, temperature=temperature)

    return {"participant": participant, "n": n, "metrics": metrics}


def eval_test_all_participants(
    df_test: pd.DataFrame,
    model,
    scaler,
    device,
    *,
    participants=("J", "F", "Y"),
    temperature=1.0,
    batch_size=256,
    require_consistent=True,
) -> pd.DataFrame:
    """
    Возвращает таблицу метрик по каждому participant.
    """
    rows = []
    for p in participants:
        out = eval_test_for_participant(
            df_test, p, model, scaler, device,
            temperature=temperature,
            batch_size=batch_size,
            require_consistent=require_consistent,
        )
        n = out["n"]
        m = out["metrics"]
        if n == 0 or m is None:
            rows.append({
                "participant": p,
                "n": n,
                "loss": np.nan,
                "top1_soft": np.nan,
                "top1_hard": np.nan,
                "tie_rate": np.nan,
                "mean_margin": np.nan,
                "pwin_mean": np.nan,
                "pwin_q05": np.nan,
                "pwin_q95": np.nan,
            })
            continue

        rows.append({
            "participant": p,
            "n": n,
            "loss": m["loss"],
            "top1_soft": m["top1_acc_soft"],
            "top1_hard": m["top1_acc_hard"],
            "tie_rate": m["tie_rate"],
            "mean_margin": m["mean_margin"],
            "pwin_mean": m["pwin"]["mean"],
            "pwin_q05": m["pwin"]["q05"],
            "pwin_q95": m["pwin"]["q95"],
        })

    return pd.DataFrame(rows).sort_values("n", ascending=False)

In [7]:
def train_ranknet_quads(
    df: pd.DataFrame,
    *,
    temperature=1.0,

    test_size=0.2,      # доля TEST от всего df
    val_size=0.25,       # доля VAL от оставшегося после test

    random_state=0,
    batch_size=64,
    epochs=60,
    lr=1e-3,
    weight_decay=1e-4,
    hidden=(64, 32),
    dropout=0.1,
    device=None,
    gate_bias = 2.5,
    monitor="val_loss", #val_loss or val_top1_acc
    min_delta=0.0,
    restore_best=True,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # --- Split по индексам, чтобы легко вернуть df_test ---
    idx = np.arange(len(df))

    idx_tmp, idx_te = train_test_split(
        idx, test_size=test_size, random_state=random_state, shuffle=True
    )
    idx_tr, idx_va = train_test_split(
        idx_tmp, test_size=val_size, random_state=random_state, shuffle=True
    )

    df_tr = df.iloc[idx_tr].reset_index(drop=True)
    df_va = df.iloc[idx_va].reset_index(drop=True)
    df_te = df.iloc[idx_te].reset_index(drop=True)

    # --- arrays ---
    X1_tr, X2_tr, X3_tr, X4_tr = prepare_arrays(df_tr)
    X1_va, X2_va, X3_va, X4_va = prepare_arrays(df_va)
    X1_te, X2_te, X3_te, X4_te = prepare_arrays(df_te)

    # --- scaler fit только на TRAIN кандидатах ---
    scaler = StandardScaler()
    scaler.fit(np.vstack([X1_tr, X2_tr, X3_tr, X4_tr]))

    def tr(x): return scaler.transform(x).astype(np.float32)

    X1_tr, X2_tr, X3_tr, X4_tr = tr(X1_tr), tr(X2_tr), tr(X3_tr), tr(X4_tr)
    X1_va, X2_va, X3_va, X4_va = tr(X1_va), tr(X2_va), tr(X3_va), tr(X4_va)
    X1_te, X2_te, X3_te, X4_te = tr(X1_te), tr(X2_te), tr(X3_te), tr(X4_te)

    train_loader = DataLoader(QuadDataset(X1_tr, X2_tr, X3_tr, X4_tr),
                              batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(QuadDataset(X1_va, X2_va, X3_va, X4_va),
                              batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(QuadDataset(X1_te, X2_te, X3_te, X4_te),
                              batch_size=batch_size, shuffle=False)


    model = HardMoEReward3(
        in_dim=F_DIM,
        participant_idx=participant_idx,
        hidden=hidden,
        dropout=dropout,
    ).to(device)

    # AdamW предпочтительнее, но оставлю как у тебя Adam, если хочешь:
    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    #optim = torch.optim.AdamW([
    #{"params": model.trunk.parameters(), "lr": 9e-4},
    #{"params": model.moe.parameters(),   "lr": 1e-3},
    #{"params": [model.scale],           "lr": 1e-4} if model.scale is not None else {},
    #], weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optim, T_max=epochs, eta_min=1e-6
    )

    # --- best tracking ---
    if monitor == "val_loss":
        mode = "min"
        best_score = float("inf")
    elif monitor == "val_top1_acc":
        mode = "max"
        best_score = -float("inf")
    else:
        raise ValueError("monitor must be 'val_loss' or 'val_top1_acc'")

    best_epoch = 0
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        for x1, x2, x3, x4 in train_loader:
            x1, x2, x3, x4 = x1.to(device), x2.to(device), x3.to(device), x4.to(device)

            s1 = model(x1); s2 = model(x2); s3 = model(x3); s4 = model(x4)

            loss = loss_listwise_top1(s1, s2, s3, s4, temperature=temperature)

            optim.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

        scheduler.step()

        trm = eval_metrics(model, train_loader, device, temperature=temperature)
        vam = eval_metrics(model, val_loader, device, temperature=temperature)

        print(
            f"epoch {epoch:03d} | "
            f"train loss {trm['loss']:.4f} top1_soft {trm['top1_acc_soft']:.3f} "
            f"tie {trm['tie_rate']:.3f} margin {trm['mean_margin']:.3f} | "
            f"val loss {vam['loss']:.4f} top1_soft {vam['top1_acc_soft']:.3f} "
            f"tie {vam['tie_rate']:.3f} margin {vam['mean_margin']:.3f} | "
            f"val Pwin mean {vam['pwin']['mean']:.3f} q05 {vam['pwin']['q05']:.3f} q95 {vam['pwin']['q95']:.3f}"
        )

        if epoch % 10 == 0:
          report = eval_test_all_participants(
            df_te, model, scaler, device,
            temperature=temperature
          )
          print(report)
          test_metrics = eval_metrics(model, test_loader, device, temperature=temperature)
          print(
            f"[TEST] loss {test_metrics['loss']:.4f} top1_soft {test_metrics['top1_acc_soft']:.3f} "
            f"tie {test_metrics['tie_rate']:.3f} margin {test_metrics['mean_margin']:.3f} | "
            f"Pwin mean {test_metrics['pwin']['mean']:.3f} q05 {test_metrics['pwin']['q05']:.3f} q95 {test_metrics['pwin']['q95']:.3f}"
          )

        current = vam["loss"] if monitor == "val_loss" else vam["top1_acc_soft"]
        improved = (current < best_score - min_delta) if mode == "min" else (current > best_score + min_delta)

        if improved:
            best_score = current
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if restore_best and best_state is not None:
        model.load_state_dict(best_state)
        print(f"Restored best model from epoch {best_epoch} with {monitor}={best_score:.4f}")

    # --- финальная оценка на test ---
    test_metrics = eval_metrics(model, test_loader, device, temperature=temperature)
    print(
        f"[TEST] loss {test_metrics['loss']:.4f} top1_soft {test_metrics['top1_acc_soft']:.3f} "
        f"tie {test_metrics['tie_rate']:.3f} margin {test_metrics['mean_margin']:.3f} | "
        f"Pwin mean {test_metrics['pwin']['mean']:.3f} q05 {test_metrics['pwin']['q05']:.3f} q95 {test_metrics['pwin']['q95']:.3f}"
    )

    info = {
        "best_epoch": best_epoch,
        "best_score": best_score,
        "monitor": monitor,
        "test_metrics": test_metrics,
        "sizes": {"train": len(df_tr), "val": len(df_va), "test": len(df_te)},
        "indices": {"train": idx_tr, "val": idx_va, "test": idx_te},
    }

    splits = {"df_train": df_tr, "df_val": df_va, "df_test": df_te}
    return model, scaler, device, info, splits

In [ ]:
temp = 2.9
model, scaler, device, monitor_dict, splits = train_ranknet_quads(
    df,
    temperature=temp,
    epochs=200,
    lr=3e-4,
    weight_decay=1e-3,
    hidden=(2048, 1536),
    dropout=0.8,
    batch_size=1024,
)

epoch 001 | train loss 1.2844 top1_soft 0.545 tie 0.000 margin 0.027 | val loss 1.2927 top1_soft 0.527 tie 0.000 margin -0.002 | val Pwin mean 0.276 q05 0.224 q95 0.318
epoch 002 | train loss 1.1953 top1_soft 0.551 tie 0.000 margin 0.069 | val loss 1.2117 top1_soft 0.537 tie 0.000 margin 0.011 | val Pwin mean 0.305 q05 0.198 q95 0.394
epoch 003 | train loss 1.1193 top1_soft 0.546 tie 0.000 margin 0.118 | val loss 1.1441 top1_soft 0.542 tie 0.000 margin 0.028 | val Pwin mean 0.336 q05 0.170 q95 0.482
epoch 004 | train loss 1.0622 top1_soft 0.547 tie 0.000 margin 0.179 | val loss 1.0945 top1_soft 0.545 tie 0.000 margin 0.057 | val Pwin mean 0.366 q05 0.143 q95 0.575
epoch 005 | train loss 1.0260 top1_soft 0.548 tie 0.000 margin 0.245 | val loss 1.0645 top1_soft 0.547 tie 0.000 margin 0.091 | val Pwin mean 0.393 q05 0.120 q95 0.655
epoch 006 | train loss 1.0062 top1_soft 0.552 tie 0.000 margin 0.312 | val loss 1.0505 top1_soft 0.548 tie 0.000 margin 0.128 | val Pwin mean 0.414 q05 0.101 q

In [ ]:
df_test = splits["df_test"]

report = eval_test_all_participants(
    df_test, model, scaler, device,
    temperature=temp
)
print(report)

  participant    n      loss  top1_soft  top1_hard  tie_rate  mean_margin  \
0           J  347  1.145523   0.512968   0.512968       0.0    -0.071500   
2           Y  305  0.793214   0.649180   0.649180       0.0     0.936395   
1           F  283  1.103983   0.530035   0.530035       0.0    -0.122609   

   pwin_mean  pwin_q05  pwin_q95  
0   0.398764  0.081343  0.779628  
2   0.554686  0.106871  0.917074  
1   0.390663  0.078711  0.710720  


In [8]:
import numpy as np
import pandas as pd

import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler


def train_one_fold_quads(
    *,
    random_state = 0,
    df_train_fold: pd.DataFrame,
    df_val_fold: pd.DataFrame,
    df_test_holdout: pd.DataFrame,
    hidden=(64, 32),
    dropout=0.1,
    temperature=1.0,
    batch_size=64,
    epochs=200,
    lr=5e-4,
    weight_decay=1e-3,
    device=None,
    monitor="val_loss",     # "val_loss" или "val_top1_acc"
    min_delta=0.0,
    restore_best=True,
    fold_id,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # --- arrays ---
    X1_tr, X2_tr, X3_tr, X4_tr = prepare_arrays(df_train_fold)
    X1_va, X2_va, X3_va, X4_va = prepare_arrays(df_val_fold)
    X1_te, X2_te, X3_te, X4_te = prepare_arrays(df_test_holdout)

    # --- scaler только на TRAIN фолда ---
    scaler = StandardScaler()
    scaler.fit(np.vstack([X1_tr, X2_tr, X3_tr, X4_tr]))

    def tr(x): return scaler.transform(x).astype(np.float32)

    X1_tr, X2_tr, X3_tr, X4_tr = tr(X1_tr), tr(X2_tr), tr(X3_tr), tr(X4_tr)
    X1_va, X2_va, X3_va, X4_va = tr(X1_va), tr(X2_va), tr(X3_va), tr(X4_va)
    X1_te, X2_te, X3_te, X4_te = tr(X1_te), tr(X2_te), tr(X3_te), tr(X4_te)

    train_loader = DataLoader(QuadDataset(X1_tr, X2_tr, X3_tr, X4_tr),
                              batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(QuadDataset(X1_va, X2_va, X3_va, X4_va),
                              batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(QuadDataset(X1_te, X2_te, X3_te, X4_te),
                              batch_size=batch_size, shuffle=False)

    torch.manual_seed(random_state + fold_id)
    model = HardMoEReward3(
        in_dim=F_DIM,
        participant_idx=participant_idx,
        hidden=hidden,
        dropout=dropout,
    ).to(device)


    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs, eta_min=1e-6)

    # --- best tracking ---
    if monitor == "val_loss":
        mode = "min"
        best_score = float("inf")
    elif monitor == "val_top1_acc":
        mode = "max"
        best_score = -float("inf")
    else:
        raise ValueError("monitor must be 'val_loss' or 'val_top1_acc'")

    best_epoch = 0
    best_state = None
    no_improve = 0

    for epoch in range(1, epochs + 1):
        model.train()
        for x1, x2, x3, x4 in train_loader:
            x1, x2, x3, x4 = x1.to(device), x2.to(device), x3.to(device), x4.to(device)

            s1 = model(x1); s2 = model(x2); s3 = model(x3); s4 = model(x4)
            loss = loss_listwise_top1(s1, s2, s3, s4, temperature=temperature)

            optim.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

        scheduler.step()

        trm = eval_metrics(model, train_loader, device, temperature=temperature)
        vam = eval_metrics(model, val_loader, device, temperature=temperature)

        # мониторим
        current = vam["loss"] if monitor == "val_loss" else vam["top1_acc_soft"]
        improved = (current < best_score - min_delta) if mode == "min" else (current > best_score + min_delta)

        if improved:
            best_score = float(current)
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if epoch == 1 or epoch % 25 == 0 or epoch == epochs:
            print(
                f"epoch {epoch:03d} | "
                f"train loss {trm['loss']:.4f} top1 {trm['top1_acc_soft']:.3f} | "
                f"val loss {vam['loss']:.4f} top1 {vam['top1_acc_soft']:.3f}"
            )

    if restore_best and best_state is not None:
        model.load_state_dict(best_state)

    # --- финальные метрики ---
    val_metrics  = eval_metrics(model, val_loader, device, temperature=temperature)
    test_metrics = eval_metrics(model, test_loader, device, temperature=temperature)

    info = {
        "best_epoch": best_epoch,
        "best_score": best_score,
        "monitor": monitor,
        "sizes": {"train": len(df_train_fold), "val": len(df_val_fold), "test": len(df_test_holdout)},
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
    }

    return model, scaler, device, info


def kfold_cv_ranknet_quads(
    df: pd.DataFrame,
    *,
    k=5,
    test_size=0.2,          # общий holdout TEST от всего df
    random_state=0,
    shuffle=True,
    hidden=(64, 32),
    dropout=0.1,
    temperature=1.0,
    batch_size=64,
    epochs=200,
    lr=5e-4,
    weight_decay=1e-3,
    device=None,
    monitor="val_loss",
    min_delta=0.0,
    restore_best=True,
):
    # --- 1) общий TEST holdout ---
    idx = np.arange(len(df))
    idx_trainval, idx_test = train_test_split(
        idx, test_size=test_size, random_state=random_state, shuffle=True
    )

    df_test = df.iloc[idx_test].reset_index(drop=True)
    df_trainval = df.iloc[idx_trainval].reset_index(drop=True)

    # индексы внутри trainval
    idx_tv = np.arange(len(df_trainval))

    kf = KFold(n_splits=k, shuffle=shuffle, random_state=random_state)

    fold_rows = []
    fold_models = []
    fold_scalers = []
    fold_splits = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(idx_tv), start=1):
        print(f"\n=== Fold {fold}/{k} | train={len(tr_idx)} val={len(va_idx)} | TEST holdout={len(df_test)} ===")

        df_tr = df_trainval.iloc[tr_idx].reset_index(drop=True)
        df_va = df_trainval.iloc[va_idx].reset_index(drop=True)

        model, scaler, device_used, info = train_one_fold_quads(
            df_train_fold=df_tr,
            df_val_fold=df_va,
            df_test_holdout=df_test,
            hidden=hidden,
            dropout=dropout,
            temperature=temperature,
            batch_size=batch_size,
            epochs=epochs,
            lr=lr,
            weight_decay=weight_decay,
            device=device,
            monitor=monitor,
            min_delta=min_delta,
            restore_best=restore_best,
            fold_id=fold,
        )

        row = {
            "fold": fold,
            "best_epoch": info["best_epoch"],
            "best_score": info["best_score"],
            "monitor": info["monitor"],
            "n_train": info["sizes"]["train"],
            "n_val": info["sizes"]["val"],
            "n_test": info["sizes"]["test"],
            # val:
            "val_loss": info["val_metrics"]["loss"],
            "val_top1": info["val_metrics"]["top1_acc_soft"],
            "val_margin": info["val_metrics"]["mean_margin"],
            # test:
            "test_loss": info["test_metrics"]["loss"],
            "test_top1": info["test_metrics"]["top1_acc_soft"],
            "test_margin": info["test_metrics"]["mean_margin"],
        }
        fold_rows.append(row)

        fold_models.append(model)
        fold_scalers.append(scaler)
        fold_splits.append({"df_train": df_tr, "df_val": df_va, "df_test": df_test})

        print(f"Fold {fold} | val_top1={row['val_top1']:.3f} test_top1={row['test_top1']:.3f} best epoch={row['best_epoch']}")

    results = pd.DataFrame(fold_rows)

    summary = {
        "val_top1_mean": float(results["val_top1"].mean()),
        "val_top1_std": float(results["val_top1"].std(ddof=1)) if k > 1 else 0.0,
        "test_top1_mean": float(results["test_top1"].mean()),
        "test_top1_std": float(results["test_top1"].std(ddof=1)) if k > 1 else 0.0,
        "val_loss_mean": float(results["val_loss"].mean()),
        "test_loss_mean": float(results["test_loss"].mean()),
    }

    print("\n=== CV Summary (with fixed TEST holdout) ===")
    print(f"VAL  top1:  mean={summary['val_top1_mean']:.3f} std={summary['val_top1_std']:.3f}")
    print(f"TEST top1:  mean={summary['test_top1_mean']:.3f} std={summary['test_top1_std']:.3f}")
    print(f"VAL  loss:  mean={summary['val_loss_mean']:.4f}")
    print(f"TEST loss:  mean={summary['test_loss_mean']:.4f}")

    info = {
        "idx": {"trainval": idx_trainval, "test": idx_test},
        "df_test": df_test,
        "results": results,
        "summary": summary,
        "models": fold_models,
        "scalers": fold_scalers,
        "splits": fold_splits,
    }
    return info

In [9]:
temp = 1.9
cv = kfold_cv_ranknet_quads(
    df,
    k=4,
    hidden=(2048, 1536),
    dropout=0.8,
    temperature=temp,
    batch_size=512,
    epochs=200,
    lr=3e-4,
    weight_decay=1e-4,
    monitor="val_loss",
)

cv["results"]   # таблица по фолдам
cv["summary"]   # среднее/стд


=== Fold 1/4 | train=2805 val=935 | TEST holdout=935 ===
epoch 001 | train loss 1.1699 top1 0.552 | val loss 1.1900 top1 0.540
epoch 025 | train loss 0.8964 top1 0.635 | val loss 1.0037 top1 0.556
epoch 050 | train loss 0.8465 top1 0.663 | val loss 1.0013 top1 0.559
epoch 075 | train loss 0.8098 top1 0.679 | val loss 1.0043 top1 0.559
epoch 100 | train loss 0.7832 top1 0.694 | val loss 1.0093 top1 0.557
epoch 125 | train loss 0.7628 top1 0.702 | val loss 1.0120 top1 0.563
epoch 150 | train loss 0.7519 top1 0.710 | val loss 1.0139 top1 0.557
epoch 175 | train loss 0.7480 top1 0.712 | val loss 1.0149 top1 0.561
epoch 200 | train loss 0.7471 top1 0.712 | val loss 1.0152 top1 0.561
Fold 1 | val_top1=0.558 test_top1=0.555 best epoch=38

=== Fold 2/4 | train=2805 val=935 | TEST holdout=935 ===
epoch 001 | train loss 1.1866 top1 0.544 | val loss 1.1851 top1 0.536
epoch 025 | train loss 0.9141 top1 0.620 | val loss 0.9677 top1 0.583
epoch 050 | train loss 0.8661 top1 0.645 | val loss 0.9599 t

{'val_top1_mean': 0.5796791443850267,
 'val_top1_std': 0.016752000177498345,
 'test_top1_mean': 0.5537433155080214,
 'test_top1_std': 0.007377612152076576,
 'val_loss_mean': 0.9728679931737522,
 'test_loss_mean': 1.012478805942969}

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
df_test = cv["df_test"]

reports = []

for i, (model, scaler) in enumerate(zip(cv["models"], cv["scalers"]), start=1):
    r = eval_test_all_participants(
        df_test,
        model,
        scaler,
        device,
        temperature=temp,
    )
    r["fold"] = i
    reports.append(r)

reports_df = pd.concat(reports, ignore_index=False)
for i in range(len(reports)):
  print(reports[i])

  participant    n      loss  top1_soft  top1_hard  tie_rate  mean_margin  \
0           J  347  1.134361   0.507205   0.507205       0.0    -0.040976   
2           Y  305  0.779610   0.649180   0.649180       0.0     0.916224   
1           F  283  1.117142   0.512367   0.512367       0.0    -0.127727   

   pwin_mean  pwin_q05  pwin_q95  fold  
0   0.381496  0.097517  0.723487     1  
2   0.552535  0.104826  0.898901     1  
1   0.374150  0.105849  0.666127     1  
  participant    n      loss  top1_soft  top1_hard  tie_rate  mean_margin  \
0           J  347  1.146266   0.489914   0.489914       0.0    -0.061419   
2           Y  305  0.778981   0.652459   0.652459       0.0     0.952307   
1           F  283  1.110123   0.508834   0.508834       0.0    -0.111584   

   pwin_mean  pwin_q05  pwin_q95  fold  
0   0.382509  0.090780  0.741765     2  
2   0.556372  0.100583  0.909274     2  
1   0.376265  0.105085  0.659880     2  
  participant    n      loss  top1_soft  top1_hard  ti

In [12]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader


def _summary_stats_1d(t: torch.Tensor) -> dict:
    """t: (N,) on any device"""
    t = t.detach()
    q05, q50, q95 = torch.quantile(t, torch.tensor([0.05, 0.50, 0.95], device=t.device))
    return {
        "mean": float(t.mean().item()),
        "std": float(t.std(unbiased=False).item()),
        "q05": float(q05.item()),
        "q50": float(q50.item()),
        "q95": float(q95.item()),
        "min": float(t.min().item()),
        "max": float(t.max().item()),
    }


@torch.no_grad()
def ensemble_eval_metrics_quads(
    df,
    models,
    scalers,
    device,
    *,
    temperature: float = 1.0,
    batch_size: int = 256,
    tie_eps: float = 1e-6,
):
    """
    Ensemble метрики для четверок:
      - усредняем скоры s1..s4 по моделям (каждая со своим scaler)
      - считаем listwise CE loss (winner = x1)
      - считаем top1_hard, top1_soft, tie_rate, margin, pwin summary

    Возвращает словарь в стиле твоего eval_metrics.
    """
    assert len(models) == len(scalers), "models and scalers must have same length"
    m = len(models)
    if m == 0:
        raise ValueError("Empty models list")

    # raw arrays
    X1, X2, X3, X4 = prepare_arrays(df)

    # loader по raw (без нормализации)
    ds = QuadDataset(
        X1.astype(np.float32),
        X2.astype(np.float32),
        X3.astype(np.float32),
        X4.astype(np.float32),
    )
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)

    total = 0
    total_loss = 0.0
    top1_correct = 0
    top1_soft_sum = 0.0
    tie_count = 0
    margin_sum = 0.0

    pwin_all = []

    for x1, x2, x3, x4 in loader:
        bs = x1.size(0)

        # усредняемые скоры на device
        s1_mean = torch.zeros(bs, device=device)
        s2_mean = torch.zeros(bs, device=device)
        s3_mean = torch.zeros(bs, device=device)
        s4_mean = torch.zeros(bs, device=device)

        # CPU -> numpy для scaler
        x1_np = x1.numpy()
        x2_np = x2.numpy()
        x3_np = x3.numpy()
        x4_np = x4.numpy()

        for model, scaler in zip(models, scalers):
            model.eval()

            x1n = torch.from_numpy(scaler.transform(x1_np).astype(np.float32)).to(device)
            x2n = torch.from_numpy(scaler.transform(x2_np).astype(np.float32)).to(device)
            x3n = torch.from_numpy(scaler.transform(x3_np).astype(np.float32)).to(device)
            x4n = torch.from_numpy(scaler.transform(x4_np).astype(np.float32)).to(device)

            s1_mean += model(x1n)
            s2_mean += model(x2n)
            s3_mean += model(x3n)
            s4_mean += model(x4n)

        s1_mean /= m
        s2_mean /= m
        s3_mean /= m
        s4_mean /= m

        # logits для listwise loss / pwin
        logits = torch.stack([s1_mean, s2_mean, s3_mean, s4_mean], dim=1) / temperature  # (B,4)
        target = torch.zeros(bs, dtype=torch.long, device=device)

        loss = F.cross_entropy(logits, target)
        total_loss += float(loss.item()) * bs
        total += bs

        # hard top1
        pred = logits.argmax(dim=1)
        top1_correct += int((pred == 0).sum().item())

        # margin
        best_loser = torch.max(torch.stack([s2_mean, s3_mean, s4_mean], dim=1), dim=1).values
        margin = (s1_mean - best_loser)  # (B,)
        margin_sum += float(margin.mean().item()) * bs

        # tie: если margin очень мал (победитель ≈ лучший проигравший)
        tie_count += int((margin.abs() < tie_eps).sum().item())

        # top1_soft: “мягкая уверенность” победы через sigmoid(margin)
        # (это обычно ближе к твоим top1_soft, чем pwin=softmax)
        top1_soft_sum += float(torch.sigmoid(margin).mean().item()) * bs

        # p(win): softmax вероятность winner среди 4
        pwin = torch.softmax(logits, dim=1)[:, 0]  # (B,)
        pwin_all.append(pwin.detach().cpu())

    pwin_all = torch.cat(pwin_all, dim=0) if len(pwin_all) else torch.empty(0)

    return {
        "loss": total_loss / max(total, 1),
        "top1_acc_soft": top1_soft_sum / max(total, 1),
        "top1_acc_hard": top1_correct / max(total, 1),
        "tie_rate": tie_count / max(total, 1),
        "mean_margin": margin_sum / max(total, 1),
        "pwin": _summary_stats_1d(pwin_all) if pwin_all.numel() else None,
        "n": total,
    }

In [17]:
device = "cuda" if torch.cuda.is_available() else "cpu"
df_part = cv["df_test"]
df_part = df_part[df_part['x1_participant_J']==1]
ens_test = ensemble_eval_metrics_quads(
    df_part,
    cv["models"],
    cv["scalers"],
    device,
    temperature=temp,
    batch_size=256,
)

print("[ENSEMBLE TEST]", ens_test)

[ENSEMBLE TEST] {'loss': 1.133406207609589, 'top1_acc_soft': 0.4988184097177357, 'top1_acc_hard': 0.4956772334293948, 'tie_rate': 0.0, 'mean_margin': -0.04498690094594825, 'pwin': {'mean': 0.3852481544017792, 'std': 0.20362021028995514, 'q05': 0.09395086765289307, 'q50': 0.37223759293556213, 'q95': 0.7426303029060364, 'min': 0.016440941020846367, 'max': 0.9017712473869324}, 'n': 347}
